# Baseball Data Wrangling and Next Season Prediction
Ngee Ann Polytechnic academic project. See README.md for setup, source data and limitations.

In [ ]:
# import the packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

pd.set_option('display.max_columns', None)

### Step 1: Load and Explore the Data

In [ ]:
# load the three required tables
data_path = './datasets/csv/'

batting = pd.read_csv(data_path + 'batting.csv')
player = pd.read_csv(data_path + 'player.csv')
all_star = pd.read_csv(data_path + 'all_star.csv')

In [ ]:
# inspect each complete source table
tables = [batting, player, all_star]
table_names = ['Batting', 'Player', 'All-Star']

count = 0

for table in tables:
    print(table_names[count])
    print('Shape:', table.shape)
    display(table.head())
    print()
    count = count + 1


In [ ]:
# discover missing values before making a decision
count = 0

for table in tables:
    print(table_names[count])
    display(table.isnull().mean())
    print()
    count = count + 1

In [ ]:
# check repeated player-season records
print('Repeated batting player-year rows:')
print(batting.duplicated(subset=['player_id', 'year']).sum())

print('Repeated All-Star player-year rows:')
print(all_star.duplicated(subset=['player_id', 'year']).sum())

print('Batting hand categories:')
display(player['bats'].value_counts(dropna=False))

### Step 2: Data Wrangling on multiple tables 

In [ ]:
batting_variables = ['g', 'ab', 'r', 'h', 'double','triple', 'hr', 'rbi', 'bb', 'so']
batting_selected = batting[['player_id', 'year', 'g', 'ab', 'r', 'h', 'double','triple', 'hr', 'rbi', 'bb', 'so']]

print('Rows before grouping:', len(batting_selected))
print(batting_selected.head())


In [ ]:
# add team stints to create one row per player-season
batting_season = pd.DataFrame()
batting_season[batting_variables] = batting_selected.groupby(['player_id', 'year'])[batting_variables].sum()

# count recorded values so an entirely missing season does not become a false zero
batting_count = pd.DataFrame()
batting_count[batting_variables] = batting_selected.groupby(['player_id', 'year'])[batting_variables].count()


for variable in batting_variables:
    batting_season[variable] = np.where(batting_count[variable] == 0,np.nan,batting_season[variable])

batting_season.reset_index(inplace=True)

print('Rows after grouping:', len(batting_season))
display(batting_season.head())

In [ ]:
# inspect missing values after player-season grouping
print(batting_season.isnull().mean())

In [ ]:
# create the following-season target table
next_season = batting_season[['player_id', 'year', 'hr']].copy()
next_season['next_record'] = 1
next_season['year'] = next_season['year'] - 1
next_season['next_hr'] = next_season['hr']
next_season = next_season[['player_id', 'year', 'next_hr', 'next_record']]

print(next_season.head())


In [ ]:
# one All-Star indicator per player-season
all_star['all_star'] = 1

all_star_season = pd.DataFrame()
all_star_season['all_star'] = all_star.groupby(['player_id', 'year'])['all_star'].max()

all_star_season.reset_index(inplace=True)
print(all_star_season.head())


In [ ]:
# 2015 cannot be a current predictor because 2016 is unavailable
model_data = batting_season[batting_season['year'] <= 2014]

# merge the following-season target
model_data = pd.merge(model_data,next_season,on=['player_id', 'year'],how='left')

print('Rows before target cleaning:', len(model_data))

In [ ]:
print('Missing target percentages:')
print(model_data[['next_hr', 'next_record']].isnull().mean() * 100)


In [ ]:
# remove rows without a usable following-season target
model_data = model_data.dropna(subset=['next_record', 'next_hr'])
# remove the temporary matching indicator
model_data.drop('next_record',axis=1,inplace=True)


In [ ]:
print('Rows before 1933:', (model_data['year'] < 1933).sum())
print('Percentage before 1933:', round((model_data['year'] < 1933).mean() * 100, 2), '%')

In [ ]:
# remove seasons before All-Star records became available
model_data = model_data[model_data['year'] >= 1933]

print('Rows after removing pre-1933 seasons:', len(model_data))

In [ ]:
# merge the All-Star table
model_data = pd.merge(model_data,all_star_season,on=['player_id', 'year'],how='left')

# no matching All-Star row means the player was not selected
model_data['all_star'] = model_data['all_star'].fillna(0)

In [ ]:
# merge player information and calculate age
player_selected = player[['player_id', 'birth_year', 'bats']]

model_data = pd.merge(model_data,player_selected,on='player_id',how='left')

model_data['age'] = model_data['year'] - model_data['birth_year']
model_data = model_data[['g', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'bb', 'so', 'age', 'bats', 'all_star', 'next_hr']]

print('Rows after wrangling:', len(model_data))
display(model_data.head())


### Step 3: Data Cleansing and Transformation

In [ ]:
print(model_data.isnull().mean())

In [ ]:
numerical_variables = ['g', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'bb', 'so', 'age']
display(model_data[numerical_variables + ['next_hr']].describe())

plt.figure(figsize=(8, 4))
sns.histplot(model_data['next_hr'], bins=30)
plt.title('Following-Season Home Runs')
plt.xlabel('Home Runs')
plt.show()

In [ ]:
model_data[numerical_variables].hist(bins=30, figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
for variable in numerical_variables:
    plt.figure(figsize=(3, 6))
    sns.boxplot(y=model_data[variable])
    plt.title(variable)
    plt.show()

In [ ]:
model_data = model_data[model_data['age'] >= 0]

In [ ]:
# split before learning imputation values and caps
X_train, X_test, y_train, y_test = train_test_split(model_data.drop('next_hr', axis=1),model_data['next_hr'],test_size=0.25,random_state=0)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
imputer = SimpleImputer(strategy='median')

X_train[numerical_variables] = imputer.fit_transform(X_train[numerical_variables])
X_test[numerical_variables] = imputer.transform(X_test[numerical_variables])

In [ ]:
bats_imputer = SimpleImputer(strategy='most_frequent')

X_train[['bats']] = bats_imputer.fit_transform(X_train[['bats']])
X_test[['bats']] = bats_imputer.transform(X_test[['bats']])


In [ ]:
print('Train missing values:', X_train.isnull().sum().sum())
print('Test missing values:', X_test.isnull().sum().sum())

In [ ]:
X_train = pd.get_dummies(X_train, columns=['bats'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['bats'], drop_first=True)

print(X_train.shape)
print(X_test.shape)
display(X_train.head())

In [ ]:
X_train_original = X_train.copy()
X_test_original = X_test.copy()

In [ ]:
X_train_capped = X_train_original.copy()
X_test_capped = X_test_original.copy()

In [ ]:
capping_variables = ['r', 'h', 'double', 'triple', 'hr', 'rbi', 'bb', 'so', 'age']

for variable in capping_variables:
    lower_limit = X_train_original[variable].quantile(0.05)
    upper_limit = X_train_original[variable].quantile(0.95)

    X_train_capped[variable] = np.where(X_train_capped[variable] < lower_limit, lower_limit, X_train_capped[variable])
    X_train_capped[variable] = np.where(X_train_capped[variable] > upper_limit, upper_limit, X_train_capped[variable])

    X_test_capped[variable] = np.where(X_test_capped[variable] < lower_limit, lower_limit, X_test_capped[variable])
    X_test_capped[variable] = np.where(X_test_capped[variable] > upper_limit, upper_limit, X_test_capped[variable])

In [ ]:
X_train_transformed = X_train_capped.copy()
X_test_transformed = X_test_capped.copy()

In [ ]:
power_variables = ['r', 'h', 'double', 'triple', 'hr', 'rbi', 'bb', 'so']

X_train_yeo = X_train_original.copy()
X_test_yeo = X_test_original.copy()

power_transformer_yeo = PowerTransformer(method='yeo-johnson', standardize=False)

X_train_yeo[power_variables] = power_transformer_yeo.fit_transform(X_train_original[power_variables])
X_test_yeo[power_variables] = power_transformer_yeo.transform(X_test_original[power_variables])

In [ ]:

power_transformer = PowerTransformer(method='yeo-johnson', standardize=False)

X_train_transformed[power_variables] = power_transformer.fit_transform(X_train_capped[power_variables])
X_test_transformed[power_variables] = power_transformer.transform(X_test_capped[power_variables])

print(power_transformer.lambdas_)

In [ ]:
scaler_original = StandardScaler()
X_train_original[numerical_variables] = scaler_original.fit_transform(X_train_original[numerical_variables])
X_test_original[numerical_variables] = scaler_original.transform(X_test_original[numerical_variables])

scaler_capped = StandardScaler()
X_train_capped[numerical_variables] = scaler_capped.fit_transform(X_train_capped[numerical_variables])
X_test_capped[numerical_variables] = scaler_capped.transform(X_test_capped[numerical_variables])

scaler_transformed = StandardScaler()
X_train_transformed[numerical_variables] = scaler_transformed.fit_transform(X_train_transformed[numerical_variables])
X_test_transformed[numerical_variables] = scaler_transformed.transform(X_test_transformed[numerical_variables])

scaler_yeo = StandardScaler()

X_train_yeo[numerical_variables] = scaler_yeo.fit_transform(X_train_yeo[numerical_variables])
X_test_yeo[numerical_variables] = scaler_yeo.transform(X_test_yeo[numerical_variables])

### Step 4: Machine Learning Modelling

In [ ]:
model_original = LinearRegression()
model_original.fit(X_train_original, y_train)

model_capped = LinearRegression()
model_capped.fit(X_train_capped, y_train)

model_transformed = LinearRegression()
model_transformed.fit(X_train_transformed, y_train)

model_yeo = LinearRegression()
model_yeo.fit(X_train_yeo, y_train)



In [ ]:
y_train_preds_original = model_original.predict(X_train_original)
y_test_preds_original = model_original.predict(X_test_original)

y_train_preds_capped = model_capped.predict(X_train_capped)
y_test_preds_capped = model_capped.predict(X_test_capped)

y_train_preds_transformed = model_transformed.predict(X_train_transformed)
y_test_preds_transformed = model_transformed.predict(X_test_transformed)

y_train_preds_yeo = model_yeo.predict(X_train_yeo)
y_test_preds_yeo = model_yeo.predict(X_test_yeo)


In [ ]:
mean_target_value = y_train.mean()
print('Mean target value:', round(mean_target_value, 2))

y_train_preds_naive = [round(mean_target_value, 2) for row in X_train.iloc[:, 0]]
y_test_preds_naive = [round(mean_target_value, 2) for row in X_test.iloc[:, 0]]

In [ ]:
# Naive baseline
train_rmse_naive = np.sqrt(mean_squared_error(y_train, y_train_preds_naive))
test_rmse_naive = np.sqrt(mean_squared_error(y_test, y_test_preds_naive))
train_r2_naive = r2_score(y_train, y_train_preds_naive)
test_r2_naive = r2_score(y_test, y_test_preds_naive)

# Original
train_rmse_original = np.sqrt(mean_squared_error(y_train, y_train_preds_original))
test_rmse_original = np.sqrt(mean_squared_error(y_test, y_test_preds_original))
train_r2_original = r2_score(y_train, y_train_preds_original)
test_r2_original = r2_score(y_test, y_test_preds_original)

# Capped
train_rmse_capped = np.sqrt(mean_squared_error(y_train, y_train_preds_capped))
test_rmse_capped = np.sqrt(mean_squared_error(y_test, y_test_preds_capped))
train_r2_capped = r2_score(y_train, y_train_preds_capped)
test_r2_capped = r2_score(y_test, y_test_preds_capped)

# Yeo-Johnson
train_rmse_yeo = np.sqrt(mean_squared_error(y_train, y_train_preds_yeo))
test_rmse_yeo = np.sqrt(mean_squared_error(y_test, y_test_preds_yeo))
train_r2_yeo = r2_score(y_train, y_train_preds_yeo)
test_r2_yeo = r2_score(y_test, y_test_preds_yeo)

# Capped and Yeo-Johnson
train_rmse_transformed = np.sqrt(mean_squared_error(y_train, y_train_preds_transformed))
test_rmse_transformed = np.sqrt(mean_squared_error(y_test, y_test_preds_transformed))
train_r2_transformed = r2_score(y_train, y_train_preds_transformed)
test_r2_transformed = r2_score(y_test, y_test_preds_transformed)

model_comparison = pd.DataFrame()

model_comparison['Model'] = ['Naive baseline', 'Original', 'Capped', 'Yeo-Johnson', 'Capped and Yeo-Johnson']
model_comparison['Train RMSE'] = [train_rmse_naive, train_rmse_original, train_rmse_capped, train_rmse_yeo, train_rmse_transformed]
model_comparison['Test RMSE'] = [test_rmse_naive, test_rmse_original, test_rmse_capped, test_rmse_yeo, test_rmse_transformed]
model_comparison['Train R2'] = [train_r2_naive, train_r2_original, train_r2_capped, train_r2_yeo, train_r2_transformed]
model_comparison['Test R2'] = [test_r2_naive, test_r2_original, test_r2_capped, test_r2_yeo, test_r2_transformed]

display(model_comparison.round(4))

## Report

## Report

### 1. Table of Contents

1. [Introduction with Value-Based Problem Statement](#2-introduction-with-value-based-problem-statement)
2. [Problem Formulation](#3-problem-formulation)
3. [Data Wrangling on Multiple Tables](#4-data-wrangling-on-multiple-tables)
4. [Data Cleansing and Transformation](#5-data-cleansing-and-transformation)
5. [Machine Learning Model](#6-machine-learning-model)
6. [Summary and Further Improvements](#7-summary-and-further-improvements)
7. [Reflection](#8-reflection)

---

### 2. Introduction with Value-Based Problem Statement

Home-run performance can change greatly between seasons. Being able to estimate a player’s following-season home runs may help baseball teams with player development, lineup planning and scouting. However, current-season statistics are stored across different tables and cannot be used directly for prediction.

This project investigates the question: **Can a player’s current batting performance, age, batting hand and All-Star selection predict how many home runs the player will hit in the following season?**

The aim was to combine relevant information into one player-season table, prepare the data using suitable wrangling techniques and build a Linear Regression model. The model was compared with a naïve baseline to determine whether the prepared features provided useful predictive information.

---

### 3. Problem Formulation

#### Load and explore the data

The supplied baseball database contains 26 tables. I selected three tables because they directly supported the prediction problem.

| Table          |           Original shape | Information used                                          |
| -------------- | -----------------------: | --------------------------------------------------------- |
| `batting.csv`  | 101,332 rows, 22 columns | Current batting statistics and following-season home runs |
| `player.csv`   |  18,846 rows, 24 columns | Birth year and batting hand                               |
| `all_star.csv` |    5,069 rows, 8 columns | Whether the player was selected as an All-Star            |

I inspected the shape, first five rows, data types and missing-value proportions of each table. The batting table contained 7,434 repeated player-year records. This happens when a player represents more than one team in the same season. The All-Star table contained 201 repeated player-year records because a player could have more than one All-Star record in a season. Therefore, both tables had to be grouped before merging.

The selected batting variables were games (`g`), at-bats (`ab`), runs (`r`), hits (`h`), doubles, triples, home runs (`hr`), runs batted in (`rbi`), walks (`bb`) and strikeouts (`so`). These variables describe playing opportunity, general batting production, extra-base power and plate discipline. Age, batting hand and All-Star selection added player context.

Pitching, fielding, salary and award tables were not included because they were less directly related to predicting a batter’s following-season home runs. Using more tables would also add unnecessary columns, missing values and merging complexity without a clear reason.

#### Prediction problem

This was formulated as a regression problem:

* **Predictors:** current-season batting statistics, age, batting hand and All-Star selection.
* **Target:** the number of home runs recorded by the same player in the following season.
* **One row:** one player in one season.

The target ranged from 0 to 73 home runs, with a median of 1 and a mean of approximately 4.72. Its histogram was strongly right-skewed because many player-seasons had few or no home runs, while a smaller number contained very high totals.

---

### 4. Data Wrangling on Multiple Tables

I first selected only the columns required for the prediction problem. The batting data could not be merged immediately because some players had multiple rows in the same year. I grouped it by `player_id` and `year`, then summed the batting statistics across team stints. This reduced 101,332 batting records to 93,898 unique player-season records.

When Pandas sums a group containing only missing values, the result can incorrectly appear as zero. I therefore created `batting_count`, which counted the recorded values for every selected variable in each player-season. If the count was zero, the grouped result was changed back to `NaN`. This preserved the difference between a genuine zero and an unavailable value.

To create the target, I copied `player_id`, `year` and `hr` into a helper table called `next_season`. The year in this helper table was reduced by one. For example, a player’s home runs in 2011 were relabelled as the target for the player’s 2010 predictor row. The `next_record` column was set to 1 as a temporary matching indicator. After a left merge, it showed whether a real following-season record had been found.

Rows without both `next_record` and `next_hr` were removed. I did not impute the target because doing so would invent the answer that the model was supposed to learn. A missing following-season record could mean retirement, leaving Major League Baseball or unavailable data, so it was not automatically treated as zero.

All-Star records began in 1933. Before this year, a missing All-Star record did not mean the player was not selected because the event did not yet exist. Therefore, 15,076 pre-1933 player-seasons were removed instead of being imputed. From 1933 onward, unmatched All-Star rows were filled with zero because they genuinely represented players who were not selected. Repeated All-Star records were grouped using `max()`, producing one binary indicator per player-season.

Finally, player information was merged using `player_id`. Age was calculated as season year minus birth year. The final wrangled table contained **50,127 player-seasons** and 14 columns, including the target.

---

### 5. Data Cleansing and Transformation

#### Missing values

After the tables were merged, approximately 1.95% of the selected batting values were missing. The target, games, age, All-Star indicator and batting hand had no missing values in the final table.

I split the data into 75% training data and 25% testing data before learning any cleaning values. This prevented the testing data from influencing the preparation of the training data.

Numerical missing values were imputed using the training median. The median was selected because the batting variables were skewed and contained extreme values, which could pull the mean upward. The median learned from the training data was then applied to the testing data. A most-frequent imputer was also included for batting hand. It made no changes to the final filtered data, but ensured that the same pipeline could handle a missing category if one appeared.

All-Star missing values were handled separately. After 1933, a missing match had a clear meaning: the player was not selected. These values were therefore filled with zero rather than using statistical imputation.

#### Categorical data

Batting hand contained three categories:

* `B`: bats with both hands
* `L`: left-handed
* `R`: right-handed

Linear Regression cannot directly use text categories. I applied one-hot encoding using `pd.get_dummies()` with `drop_first=True`. This created `bats_L` and `bats_R`, while `B` became the reference category. This avoided creating a false numerical order between the three categories. The All-Star feature was already represented correctly as 0 or 1, so it was kept unchanged.

#### Outliers and numerical transformation

The histograms showed that most batting variables were strongly right-skewed. The boxplots also marked many high values as possible outliers. However, these values may represent genuine high-performing seasons rather than data-entry errors, so I did not remove them automatically.

Instead, I investigated four versions of the predictor data:

1. Original values
2. Values capped at the 5th and 95th percentiles
3. Yeo-Johnson transformed values
4. Capped and Yeo-Johnson transformed values

Capping was tested on runs, hits, doubles, triples, home runs, RBI, walks, strikeouts and age. Yeo-Johnson was tested on the skewed batting counts because it can handle zero values, unlike logarithmic and reciprocal transformations.

StandardScaler was then fitted using the training data and applied to the testing data. This placed numerical features measured on different scales into comparable units. Every treatment was evaluated using the same split, model and measurements, allowing a fair comparison.

---

### 6. Machine Learning Model

A naïve baseline was created by predicting the training target mean of 4.75 home runs for every player-season. This represented a model that ignored all predictor information. Linear Regression models were then trained using each of the four prepared datasets.

| Model                  | Train RMSE |  Test RMSE |   Train R² |    Test R² |
| ---------------------- | ---------: | ---------: | ---------: | ---------: |
| Naïve baseline         |     8.0899 |     8.1024 |    -0.0000 |    -0.0002 |
| Original               | **4.8443** | **4.8125** | **0.6414** | **0.6471** |
| Capped                 |     4.9855 |     4.9745 |     0.6202 |     0.6230 |
| Yeo-Johnson            |     5.6913 |     5.7415 |     0.5051 |     0.4978 |
| Capped and Yeo-Johnson |     5.7458 |     5.7984 |     0.4956 |     0.4878 |

RMSE measures the typical prediction error in home runs, so a lower value is better. R² measures how much variation in following-season home runs is explained by the model, so a higher value is better.

The Linear Regression model using the original values achieved the best result. Its test RMSE of 4.8125 was approximately 40.6% lower than the baseline RMSE of 8.1024. Its test R² of 0.6471 means that the model explained approximately 64.71% of the variation in following-season home runs.

The training and testing results were also close, suggesting that the selected model generalised reasonably well to unseen data. Capping reduced performance because high batting totals contained useful information about strong hitters. Yeo-Johnson also reduced performance because compressing the large values weakened useful relationships between current and following-season performance.

Therefore, the original prepared values were selected. The investigation showed that cleaner-looking or more normally distributed data does not automatically produce a better model. One limitation is that Linear Regression can produce negative predictions even though a player cannot hit fewer than zero home runs.

---

### 7. Summary and Further Improvements

This project successfully transformed three separate baseball tables into a regression-ready player-season dataset. The main wrangling steps included selecting relevant features, grouping repeated team stints, creating a following-season target, merging three tables, interpreting missing values, imputing predictors, encoding batting hand, investigating outliers, testing a power transformation and scaling numerical features.

The selected Linear Regression model reduced test RMSE from 8.1024 for the naïve baseline to 4.8125 and achieved a test R² of 0.6471. Current batting performance therefore contains useful information for predicting following-season home runs. Testing also showed that keeping genuine extreme performances produced better results than capping or transforming them.

Several improvements could be investigated. A time-based split could train on earlier seasons and test on later seasons, which would better represent a real future prediction. Rate features such as home runs per at-bat could help separate batting ability from playing opportunity. Players with very few at-bats could also be investigated separately because their season totals may be less stable. Finally, another regression model with non-negative output could be compared with Linear Regression to avoid impossible negative home-run predictions.

---

### 8. Reflection

This assignment helped me understand that data wrangling is not simply a checklist of techniques. Every decision must depend on what the data represents. For example, a missing All-Star value after 1933 meant the player was not selected, while a missing following-season target meant that the correct answer was unknown and should not be imputed.

I learned how to extract useful columns from structured tables, group transactional records, create a time-based target and merge tables using suitable keys. I also practised median and mode imputation, one-hot encoding, outlier capping, Yeo-Johnson transformation and feature scaling using Pandas and Scikit-learn.

The most important lesson was to test a wrangling decision instead of assuming that it would improve the data. Although capping and transformation made the distributions look less extreme, they reduced model performance. If I repeated the project, I would plan the row definition and target creation earlier, use more checks after each merge and investigate additional datasets and features more systematically.
